In [ ]:
# Cell 1: Imports and global setup

import os
import gc
import time
import numpy as np
import torch

# Ensure we can import the core package
import sys
sys.path.append(".")

from core.design_space import DesignSpace
from core.truth_interface import TruthModelEvaluator
from core.gp_models import GPModelManager
from core.acquisition_functions import AcquisitionFunctionManager
from core.agent_manager import BOAgent, ResourceEvent
from core.logging_utils import LoggingManager
from core.visualization import VisualizationManager

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.double

In [ ]:
# Cell 2: Problem configuration (Ti–V–Nb–Mo–Hf–Ta–W)

# --- Design space: 7D simplex with per-element ranges ---
# Ranges: [label, min, max]
COMPONENTS = [
    ("Ti", 0.0,   0.35),
    ("V",  0.0,   0.50),
    ("Nb", 0.225, 0.675),
    ("Mo", 0.0,   0.125),
    ("Hf", 0.0,   0.125),
    ("Ta", 0.0,   0.45),
    ("W",  0.0,   0.125),
]
STEP = 0.025
SEED = 256

design_space = DesignSpace(
    components=COMPONENTS,
    step=STEP,
    seed=SEED,
)

bounds_np = design_space.get_bounds()  # shape (2, 7)
bounds_t = torch.tensor(bounds_np, dtype=DTYPE, device=DEVICE)

# --- Truth model paths (RF + scalers for this system) ---
MODEL_PATH = "models/TiVNbMoHfTaW_Max_S_Max_TCond/RFR_best_model.pkl"
X_SCALER_PATH = "models/TiVNbMoHfTaW_Max_S_Max_TCond/x_scaler.pkl"
Y_SCALER_PATH = "models/TiVNbMoHfTaW_Max_S_Max_TCond/y_scaler.pkl"

# RF outputs:
#   preds_raw[:,0] = PROP RT Therm Cond (W/(mK))
#   preds_raw[:,1] = SConf
# We decide Y[:,0] = entropy (to maximize), Y[:,1] = thermal conductivity (to maximize)

def postprocess_outputs(preds_raw: np.ndarray) -> np.ndarray:
    therm_cond = preds_raw[:, 0]
    entropy = preds_raw[:, 1]
    return np.column_stack([entropy, therm_cond])

evaluator = TruthModelEvaluator(
    model_path=MODEL_PATH,
    x_scaler_path=X_SCALER_PATH,
    y_scaler_path=Y_SCALER_PATH,
    postprocess_outputs=postprocess_outputs,
)

# --- Objective naming and scalar score (ThermCond / entropy) ---
OBJ1_NAME = "Configurational Entropy"           # Y[:,0]
OBJ2_NAME = "Therm Cond"       # Y[:,1]

def score_fn(Y):
    E = Y[:, 0]
    H = Y[:, 1]

    # Min-max normalize over current data
    E_norm = (E - E.min()) / (E.max() - E.min() + 1e-12)
    H_norm = (H - H.min()) / (H.max() - H.min() + 1e-12)

    return E_norm * H_norm

SCORE_NAME = "Normalized: ThermCond * Configurational Entropy"

# --- BO resource and loop config (same pattern as K/CTE) ---
INIT_N = 5        # initial random samples
ITERS = 100        # max BO iterations

TOTAL_BUDGET = 10100.0   # $
TOTAL_TIME = 101.0        # weeks
COST_PER_POINT = 20.0   # $ per experiment
TIME_PER_ITERATION = 1.0 # weeks per iteration

MC_SAMPLES = 256
POOL_SUBSAMPLE = 100000    # None = use full enumerated space
TOTAL_BATCH_SIZE = 5     # total points per iteration (exploit + explore)

# --- LLM / agent config ---
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
AGENT_MODEL = "gpt-4o"
MAX_REASONING_STEPS = 10

PROBLEM_DESCRIPTION = """
High Entropy Alloy / refractory alloy optimization in the Ti-V-Nb-Mo-Hf-Ta-W composition space.

Objectives:
- Maximize Configurational Entropy.
- Maximize room-temperature thermal conductivity (W/(mK)).

Input space: 7D compositions (Ti, V, Nb, Mo, Hf, Ta, W) that sum to 1.0,
with element-specific bounds and grid step 0.025.

Data is from experiments at room temperature.
"""

# --- Logging & visualization config ---
LOG_DIR = "results"
EXPERIMENT_NAME = "TiVNbMoHfTaW_Entropy-TC-Low-Start-More-Acq"
CREATE_VIS = True   # Visualization currently specialized for 5D; 7D falls back/no-op
CREATE_GIF = True

In [ ]:
# Cell 3 (Scenario C): Setup GP, acquisition, agent, logger, and time+budget events

gp_manager = GPModelManager(bounds=bounds_t, use_priors=False)
acq_manager = AcquisitionFunctionManager(bounds=bounds_t)

agent_log_dir = os.path.join(LOG_DIR, EXPERIMENT_NAME, "agent_logs")
os.makedirs(agent_log_dir, exist_ok=True)

agent = BOAgent(
    api_key=OPENAI_API_KEY,
    model=AGENT_MODEL,
    log_dir=agent_log_dir,
    max_reasoning_steps=MAX_REASONING_STEPS,
    temperature=0.7,
    problem_description=PROBLEM_DESCRIPTION,
    obj1_name=OBJ1_NAME,
    obj2_name=OBJ2_NAME,
)

events: list[ResourceEvent] = []

# Iteration 10: cap total time to 30 weeks.
# After 10 iterations (plus initial), about 11 weeks are spent, but you asked:
# "iteration 10, time left = 20" → we set time_remaining to 20.
time_remaining_after_iter10 = 20.0

events.append(
    ResourceEvent(
        iteration=10,
        event_type="time_change",
        description="Time reduced at iter 10 so that only 20 weeks remain "
                    "(total capped at ~30 weeks)",
        modifier=lambda current_time, t_rem=time_remaining_after_iter10: t_rem
    )
)

# Iteration 20: adjust budget so only 5 more iterations are possible
budget_for_five_iters = 5 * TOTAL_BATCH_SIZE * COST_PER_POINT

events.append(
    ResourceEvent(
        iteration=20,
        event_type="budget_change",
        description=f"Budget adjusted at iter 20 to allow only 5 more iterations "
                    f"(5 × {TOTAL_BATCH_SIZE} × ${COST_PER_POINT:.2f})",
        modifier=lambda current_budget, b5=budget_for_five_iters: b5
    )
)

for ev in events:
    agent.add_event(ev)

exp_dir = os.path.join(LOG_DIR, EXPERIMENT_NAME)
os.makedirs(exp_dir, exist_ok=True)
logger = LoggingManager(exp_dir, EXPERIMENT_NAME)

vis_manager = None
if CREATE_VIS:
    try:
        vis_manager = VisualizationManager(log_dir=exp_dir, design_space=design_space, experiment_name=EXPERIMENT_NAME)
    except Exception as e:
        print(f"[NB] Visualization disabled (dim != 5 or error): {e}")
        vis_manager = None

In [ ]:
# Generating plots for the objective space!
def truth_eval_Y(X: np.ndarray) -> np.ndarray:
    """
    X: (N,d) design space points.
    Returns Y: (N,2) = [obj1, obj2] from the RF truth model.
    """
    result = evaluator.evaluate(X)  # or evaluator.evaluate(X).y for your generalized version
    # For the generic TruthModelEvaluator, this is already result.y
    return result.y

import traceback

if CREATE_VIS:
    try:
        truth_plots = vis_manager.plot_objectives_from_evaluator(
            evaluator=truth_eval_Y,
            obj_names=[OBJ1_NAME, OBJ2_NAME],
            filename_prefix="truth"
        )
        print("Truth objective plots:", truth_plots)
    except Exception as e:
        print("Couldn't print objectives.")
        print("Exception type:", type(e).__name__)
        print("Exception message:", str(e))
        traceback.print_exc()


In [ ]:
# Cell 4: Agent configures GP kernel
# This is a step I originally had in case we were going to let an informed Agent tune the kernel somehow. This usually (I've never seen it not) selects default.
print("\n" + "="*80)
print("[NB] Agent configuring GP kernel...")
print("="*80)

kernel_code, kernel_reasoning = agent.configure_gp_kernel(use_priors=False)

if kernel_code:
    print("[NB] Agent requested custom kernel (not implemented here). Using default kernel.")
else:
    print("[NB] Agent selected default kernel.")

In [ ]:
# Cell 5: Initial random sampling and evaluation

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"\n[NB] Initializing with {INIT_N} random samples from Ti-V-Nb-Mo-Hf-Ta-W simplex")
X0_np = design_space.sample(INIT_N, method="random")  # (INIT_N, 7)
result0 = evaluator.evaluate(X0_np)
Y0_np = result0.y  # shape (INIT_N, 2) = [entropy, therm_cond]

X = torch.tensor(X0_np, dtype=DTYPE, device=DEVICE)
Y = torch.tensor(Y0_np, dtype=DTYPE, device=DEVICE)

# Initial resources: one iteration's worth of time
initial_budget_spent = INIT_N * COST_PER_POINT
initial_time_spent = TIME_PER_ITERATION
budget_remaining = TOTAL_BUDGET - initial_budget_spent
time_remaining = TOTAL_TIME - initial_time_spent

print(f"[NB] Initial sampling used: ${initial_budget_spent:.2f} / {initial_time_spent:.1f} weeks")
print(f"[NB] Remaining resources: ${budget_remaining:.2f} / {time_remaining:.1f} weeks")

# Log initial state
logger.log_iteration(
    iteration=0,
    X_history=X.cpu().numpy(),
    Y_history=Y.cpu().numpy(),
    n_new_points=INIT_N,
    strategy=agent,
    timing=0.0,
    extra_info={"note": "initial_sampling", "agent_kernel_reasoning": kernel_reasoning},
    acquisition_data=None,
    score_fn=score_fn,
    obj1_name=OBJ1_NAME,
    obj2_name=OBJ2_NAME,
)

logger.log_evaluations(
    iteration=0,
    X_new=X0_np,
    Y_new=Y0_np,
    selection_info=None,
    score_fn=score_fn,
    obj1_name=OBJ1_NAME,
    obj2_name=OBJ2_NAME,
)

# Typically skip visualization at iteration 0 unless your VisualizationManager handles model=None

In [ ]:
# Cell 6: Main Bayesian Optimization loop

iteration = 0

while iteration < ITERS:
    # Resource check
    if budget_remaining < COST_PER_POINT or time_remaining < TIME_PER_ITERATION:
        print(f"\n[NB] Stopping: insufficient budget (${budget_remaining:.2f}) or time ({time_remaining:.1f} weeks)")
        break

    iteration += 1
    iter_start = time.perf_counter()

    # Apply scheduled events
    budget_remaining, time_remaining, COST_PER_POINT, event_msgs = agent.process_iteration_events(
        iteration, budget_remaining, time_remaining, COST_PER_POINT
    )
    if event_msgs:
        print("\n" + "="*80)
        for msg in event_msgs:
            print(msg)
        print("="*80)

    print("\n" + "="*80)
    print(f"[NB] Iteration {iteration}/{ITERS}")
    print(f"[NB] Budget: ${budget_remaining:.2f} | Time: {time_remaining:.1f} weeks")
    print("="*80)

    # Fit GP model
    print("[NB] Fitting MultiTaskGP model...")
    model = gp_manager.fit_model(X, Y)  # Y[:,0]=entropy, Y[:,1]=therm_cond

    # Pareto front: both objectives to be maximized => negate entropy
    Y_np = Y.cpu().numpy()
    Y_for_pareto = torch.tensor(
        np.column_stack([-Y_np[:, 0], Y_np[:, 1]]),  # [-entropy, therm_cond]
        dtype=DTYPE,
        device=DEVICE,
    )
    pareto_Y, ref_point = acq_manager.compute_pareto_front(Y_for_pareto)

    # Candidate pool: full enumerated simplex or subsample
    if POOL_SUBSAMPLE is None:
        X_pool_np = design_space.space
    else:
        X_pool_np = design_space.sample(POOL_SUBSAMPLE, method="sobol")
    X_pool = torch.tensor(X_pool_np, dtype=DTYPE, device=DEVICE)

    # Acquisition options
    print("[NB] Computing acquisition options...")
    allocation_options = acq_manager.compute_all_allocation_options(
        model=model,
        X_pool=X_pool,
        pareto_Y=pareto_Y,
        ref_point=ref_point,
        mc_samples=MC_SAMPLES,
    )

    # Agent selects resource allocation
    print("[NB] Consulting LLM agent for resource allocation...")
    selected_idx, reasoning, selected_point_arrays = agent.select_resource_allocation(
        iteration=iteration,
        budget_remaining=budget_remaining,
        time_remaining=time_remaining,
        cost_per_point=COST_PER_POINT,
        time_per_point=TIME_PER_ITERATION,
        X_history=X.cpu().numpy(),
        Y_history=Y.cpu().numpy(),
        allocation_options=allocation_options,
        max_batch_size=TOTAL_BATCH_SIZE,
        score_fn=score_fn,
    )

    # Combine selected points
    if selected_point_arrays:
        X_next_np = np.vstack(selected_point_arrays)
    else:
        print("[NB] Agent selected no points; using balanced option (index 2) as fallback.")
        batch = allocation_options.qehvi_batches[2]
        parts = []
        if batch.points.size > 0:
            parts.append(batch.points)
        if batch.explore_points.size > 0:
            parts.append(batch.explore_points)
        X_next_np = np.vstack(parts) if parts else design_space.sample(1, method="random")

    print(f"[NB] Evaluating {len(X_next_np)} selected points...")
    result_next = evaluator.evaluate(X_next_np)
    Y_next_np = result_next.y  # (k, 2) [entropy, therm_cond]

    # Update datasets
    X = torch.cat([X, torch.tensor(X_next_np, dtype=DTYPE, device=DEVICE)], dim=0)
    Y = torch.cat([Y, torch.tensor(Y_next_np, dtype=DTYPE, device=DEVICE)], dim=0)

    # Resource update
    points_used = len(X_next_np)
    budget_spent = points_used * COST_PER_POINT
    time_spent = TIME_PER_ITERATION
    budget_remaining -= budget_spent
    time_remaining -= time_spent

    iter_time = time.perf_counter() - iter_start

    # Logging
    extra_info = {
        "agent_selected_option": selected_idx,
        "agent_reasoning": reasoning[:200] + "..." if len(reasoning) > 200 else reasoning,
        "budget_remaining": budget_remaining,
        "time_remaining": time_remaining,
        "budget_spent_this_iter": budget_spent,
        "time_spent_this_iter": time_spent,
    }

    logger.log_iteration(
        iteration=iteration,
        X_history=X.cpu().numpy(),
        Y_history=Y.cpu().numpy(),
        n_new_points=points_used,
        strategy=agent,
        timing=iter_time,
        extra_info=extra_info,
        acquisition_data=allocation_options,
        score_fn=score_fn,
        obj1_name=OBJ1_NAME,
        obj2_name=OBJ2_NAME,
    )

    logger.log_evaluations(
        iteration=iteration,
        X_new=X_next_np,
        Y_new=Y_next_np,
        selection_info=None,
        score_fn=score_fn,
        obj1_name=OBJ1_NAME,
        obj2_name=OBJ2_NAME,
    )

    # Visualization (currently skipped for 7D unless you extended VisualizationManager)
    if vis_manager is not None:
        vis_manager.create_iteration_plot(
            iteration=iteration,
            X_history=X.cpu().numpy(),
            Y_history=Y.cpu().numpy(),
            X_new=X_next_np,
            model=model,
            bounds=bounds_t,
            selection_info={"agent_option": selected_idx},
            score_fn=score_fn,
            score_name=SCORE_NAME,
            obj1_name=OBJ1_NAME,
            obj2_name=OBJ2_NAME,
        )

    # Print iteration summary
    scores_all = score_fn(Y.cpu().numpy())
    best_score = float(np.max(scores_all))
    print(f"\n[NB] Iteration {iteration} complete in {iter_time:.2f}s")
    print(f"[NB] Best {SCORE_NAME}: {best_score:.3f}")
    print(f"[NB] Remaining resources: ${budget_remaining:.2f} / {time_remaining:.1f} weeks")

    # Cleanup
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "="*80)
print("[NB] Optimization complete.")
print("="*80)

In [ ]:
# Cell 7: Finalization, summary, and GIF

agent.save_global_memory()
print(f"[NB] Agent memory saved in {agent_log_dir}")

logger.finalize()

if vis_manager is not None and CREATE_GIF:
    vis_manager.create_gif(duration=2.0, gif_name=f"{EXPERIMENT_NAME}.gif")

summary = logger.get_summary_stats(obj1_name=OBJ1_NAME, obj2_name=OBJ2_NAME)
print("\n[NB] Summary statistics:")
for k, v in summary.items():
    print(f"  {k}: {v}")